<h3>Extracting Features</h3>
Using ConvNeXT Large model to get the features from the image snapshots.

In [1]:
import tensorflow as tf
from tensorflow.keras.applications.convnext import ConvNeXtTiny, ConvNeXtSmall, ConvNeXtBase, ConvNeXtLarge
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.convnext import preprocess_input
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
import matplotlib.pyplot as plt

print("GPU Available: ", tf.config.list_physical_devices('GPU'))

# Initialize ConvNeXt Small model (without top classification layer)
model = ConvNeXtBase(weights='imagenet', include_top=False, pooling='avg')
model.compile()

def load_and_preprocess_image(img_path, downsampled_res):
    """Load and preprocess image for ConvNeXt"""
    img = image.load_img(img_path, target_size=downsampled_res, color_mode='grayscale')
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    return preprocess_input(img_array)

def extract_features(img_path, downsampled_res):
    """Extract features from image using ConvNeXt Small"""
    preprocessed_img = load_and_preprocess_image(img_path, downsampled_res)
    features = model.predict(preprocessed_img)
    return features.flatten()

def initial_test_run():
    # Example usage with two images
    # Replace these paths with your actual image paths
    snapshot = "data/train_data/train_images/0013.JPG"
    patch_test = "data/train_data/train_images/0014.JPG"

    # Extract features from both images
    print("Extracting features from snapshot...")
    features1 = extract_features(snapshot)

    print("Extracting features from image 2...")
    features2 = extract_features(patch_test)

    # Calculate similarity between features
    similarity = cosine_similarity([features1], [features2])[0][0]

    print(f"Feature vector size: {len(features1)}")
    print(f"Cosine similarity between images: {similarity:.4f}")

    # Higher similarity (closer to 1) suggests images might be from similar locations
    if similarity > 0.8:
        print("High similarity - likely same or nearby location")
    elif similarity > 0.6:
        print("Medium similarity - possibly related location")
    else:
        print("Low similarity - likely different locations")

    plt.figure(figsize=(10, 4))
    plt.hist(features2, bins=50, alpha=0.7, color='blue')
    plt.title("Distribution of patch_test features")
    plt.xlabel("Feature Value")
    plt.ylabel("Frequency")
    plt.show()

# initial_test_run()

GPU Available:  []




In [ ]:
def rank_patches_by_similarity(snapshot_imgs, patch_imgs, downsampleed_res):
    """Rank patches based on similarity to snapshot features"""
    similarities_per_img = {}
    
    for snapshot_img in snapshot_imgs:
        similarities = []
        print("Matching snapshot image:", snapshot_img)
        snapshot_features = extract_features(snapshot_img, downsampleed_res)
        
        for patch_index, patch_img in enumerate(patch_imgs):
            patch_features = extract_features(patch_img, downsampleed_res)
            similarity = cosine_similarity([snapshot_features], [patch_features])[0][0]
            similarities.append((patch_index, float(similarity)))  # Convert to regular Python float
        
        # Sort by similarity (descending) and get top 4 closest patches
        best_patches = sorted(similarities, key=lambda x: x[1], reverse=True)[:4]
        
        # Store results with snapshot name as key
        snapshot_name = snapshot_img.name if hasattr(snapshot_img, 'name') else str(snapshot_img)
        similarities_per_img[snapshot_name] = [
            {
                'patch_id': patch_id,
                'patch_name': patch_imgs[patch_id].name if hasattr(patch_imgs[patch_id], 'name') else str(patch_imgs[patch_id]),
                'similarity': similarity  # Already converted to float above
            }
            for patch_id, similarity in best_patches
        ]
    
    return similarities_per_img

def rank_patches_by_distance(snapshot_imgs, patch_imgs, downsampleed_res):
    """Rank patches based on euclidean distance to snapshot features"""
    distances_per_img = {}
    
    for snapshot_img in snapshot_imgs:
        distances = []
        print("Matching snapshot image:", snapshot_img)
        snapshot_features = extract_features(snapshot_img, downsampleed_res)
        
        for patch_index, patch_img in enumerate(patch_imgs):
            patch_features = extract_features(patch_img, downsampleed_res)
            distance = euclidean_distances([snapshot_features], [patch_features])[0][0]
            distances.append((patch_index, float(distance)))  # Convert to regular Python float
        
        # Sort by distance (ascending) and get top 4 closest patches
        best_patches = sorted(distances, key=lambda x: x[1], reverse=False)[:4]
        
        # Store results with snapshot name as key
        snapshot_name = snapshot_img.name if hasattr(snapshot_img, 'name') else str(snapshot_img)
        distances_per_img[snapshot_name] = [
            {
                'patch_id': patch_id,
                'patch_name': patch_imgs[patch_id].name if hasattr(patch_imgs[patch_id], 'name') else str(patch_imgs[patch_id]),
                'distance': distance  # Already converted to float above
            }
            for patch_id, distance in best_patches
        ]
    
    return distances_per_img


In [3]:
from pathlib import Path

def load_images(sample_size=5):
    """Load a sample of images from the specified folder"""
    # Define the folder path using pathlib
    folder = Path("data/train_data/train_images")

    # List image files with common extensions; note we don't sort them
    image_files = [f for f in folder.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]

    # Get a sample of file paths from the list
    sample = image_files[:sample_size]
    
    return sample

def load_test_images(sample_size=136):
    """Load all test images from the test_images folder"""
    # Define the folder path using pathlib
    folder = Path("data/test_data/test_images")

    # List image files with common extensions
    image_files = [f for f in folder.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    print(f"Found {len(image_files)} test images in {folder}")
    return image_files[len(image_files)-1]  # Return the first 'sample_size' images

def load_patches():
    """Load patch images from the specified folder"""
    # Define the folder path using pathlib
    folder = Path("map_processing/patches")

    # List image files with common extensions; note we don't sort them
    image_files = [f for f in folder.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    
    return image_files


In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

def calculate_run_accuracy(sample_size=305):
    # Load patch positions
    with open('map_processing/patches/all_patch_positions.json', 'r') as f:
        patch_data = json.load(f)
    
    # Load ground truth positions
    gt_positions = pd.read_csv('data/train_data/train_pos.csv')
    
    # Run the similarity ranking
    snapshot_imgs = load_images(sample_size)  # Load all images
    patch_imgs = load_patches()
    downsampled_res = (100, 133)
    similarities = rank_patches_by_similarity(snapshot_imgs, patch_imgs, downsampled_res)
    # similarities = rank_patches_by_distance(snapshot_imgs, patch_imgs, downsampled_res)
    results = []
    distances_under_25 = 0
    distances_under_125 = 0
    distances_under_500 = 0
    total_processed = 0
    
    print("Calculating accuracy metrics...")
    
    for snapshot_name, top_patches in similarities.items():
        # Extract image ID from filename (assuming format like "0013.JPG")
        image_id = int(snapshot_name.split('.')[0])
        
        # Find ground truth position for this image
        gt_row = gt_positions[gt_positions['id'] == image_id]
        if gt_row.empty:
            print(f"Warning: No ground truth found for image {image_id}")
            continue
            
        gt_x = gt_row.iloc[0]['x_pixel']
        gt_y = gt_row.iloc[0]['y_pixel']
        
        # Calculate predicted position from top 4 patches
        center_x_sum = 0
        center_y_sum = 0
        
        for patch_info in top_patches:
            patch_id = patch_info['patch_id']
            # Find the patch in the positions data
            patch_pos = next((p for p in patch_data['patches'] if p['patch_id'] == patch_id), None)
            if patch_pos:
                center_x_sum += patch_pos['center_x']
                center_y_sum += patch_pos['center_y']
        
        # Average the positions of top 4 patches
        pred_x = center_x_sum / len(top_patches)
        pred_y = center_y_sum / len(top_patches)
        
        # Calculate distance
        distance = np.sqrt((pred_x - gt_x)**2 + (pred_y - gt_y)**2)
        
        # Count accuracy metrics
        if distance < 25:
            distances_under_25 += 1
        if distance < 125:
            distances_under_125 += 1
        if distance < 500:
            distances_under_500 += 1
        
        total_processed += 1
        
        # Store result
        results.append({
            'image_id': image_id,
            'predicted_x': pred_x,
            'predicted_y': pred_y,
            'ground_truth_x': gt_x,
            'ground_truth_y': gt_y,
            'distance': distance
        })
        
        if total_processed % 50 == 0:
            print(f"Processed {total_processed} images...")
    
    # Create DataFrame and save to CSV
    results_df = pd.DataFrame(results)
    results_df.to_csv('accuracy_results.csv', index=False)
    
    # Calculate and print accuracy metrics
    accuracy_25 = (distances_under_25 / total_processed) * 100 if total_processed > 0 else 0
    accuracy_125 = (distances_under_125 / total_processed) * 100 if total_processed > 0 else 0
    accuracy_500 = (distances_under_500 / total_processed) * 100 if total_processed > 0 else 0
    
    print(f"\n=== ACCURACY RESULTS ===")
    print(f"Total images processed: {total_processed}")
    print(f"Accuracy within 25 pixels: {accuracy_25:.2f}% ({distances_under_25}/{total_processed})")
    print(f"Accuracy within 125 pixels: {accuracy_125:.2f}% ({distances_under_125}/{total_processed})")
    print(f"Accuracy within 500 pixels: {accuracy_500:.2f}% ({distances_under_500}/{total_processed})")
    print(f"\nMean distance error: {results_df['distance'].mean():.2f} pixels")
    print(f"Median distance error: {results_df['distance'].median():.2f} pixels")
    print(f"Results saved to 'accuracy_results.csv'")
    
    return results_df

# Calculate accuracy on a sample of the data (for testing purposes, use a smaller sample size)
accuracy_results = calculate_run_accuracy(sample_size=10)


Matching snapshot image: data\train_data\train_images\0013.JPG
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━

In [ ]:
def predict_test_coordinates():
    """Predict x,y coordinates for test images and save to CSV"""
    # Load patch positions
    with open('map_processing/patches/all_patch_positions.json', 'r') as f:
        patch_data = json.load(f)
    
    # Load test images
    test_imgs = load_test_images()
    patch_imgs = load_patches()
    downsampled_res = (100, 133)
    
    # Run the similarity ranking on test images
    similarities = rank_patches_by_similarity(test_imgs, patch_imgs, downsampled_res)
    
    results = []
    total_processed = 0
    
    print("Predicting coordinates for test images...")
    
    for test_name, best_patch in similarities.items():
        # Extract image ID from filename (assuming format like "0013.JPG")
        image_id = int(test_name.split('.')[0])
        
        # Get predicted position from the most similar patch
        patch_id = best_patch['patch_id']
        # Find the patch in the positions data
        patch_pos = next((p for p in patch_data['patches'] if p['patch_id'] == patch_id), None)
        
        if not patch_pos:
            print(f"Warning: No position data found for patch {patch_id}")
            continue
        
        pred_x = patch_pos['center_x']
        pred_y = patch_pos['center_y']
        
        total_processed += 1
        
        # Store result
        results.append({
            'id': image_id,
            'x_pixel': pred_x,
            'y_pixel': pred_y
        })
        
        if total_processed % 50 == 0:
            print(f"Processed {total_processed} test images...")
    
    # Create DataFrame and save to CSV
    results_df = pd.DataFrame(results)
    results_df.to_csv('test_predictions.csv', index=False)
    
    print(f"\n=== TEST PREDICTION RESULTS ===")
    print(f"Total test images processed: {total_processed}")
    print(f"Predictions saved to 'test_predictions.csv'")
    print(f"\nSample predictions:")
    print(results_df.head())
    
    return results_df

# Run test predictions
# test_predictions = predict_test_coordinates()